In [13]:
import pandas as pd

# read csv file that is on local 
df = pd.read_csv('posa_shaktoi.csv')

# print the first 5 rows of the dataframe
print(df.head())

   num_preg  glucose_conc  diastolic_bp  thickness  insulin   bmi  diab_pred  \
0         6           148            72         35        0  33.6      0.627   
1         1            85            66         29        0  26.6      0.351   
2         8           183            64          0        0  23.3      0.672   
3         1            89            66         23       94  28.1      0.167   
4         0           137            40         35      168  43.1      2.288   

   age  diabetes  
0   50         1  
1   31         0  
2   32         1  
3   21         0  
4   33         1  


In [7]:
# features and target variable

# columns: num_preg,glucose_conc,diastolic_bp,thickness,insulin,bmi,diab_pred,age,diabetes
X = df.drop('diabetes', axis=1)
y = df['diabetes']

In [9]:

# Section 1 – Data Exploration
# 1. What is the class balance of the target variable?

print(y.value_counts(normalize=True) * 100)

diabetes
0    500
1    268
Name: count, dtype: int64
diabetes
0    65.104167
1    34.895833
Name: proportion, dtype: float64


In [14]:
# 2. Which feature has the highest correlation with diabetes?
# let's calculate the correlation matrix
corr_matrix = df.corr()
# let's get the correlation of each feature with the target variable
target_corr = corr_matrix['diabetes'].drop('diabetes')
# let's find the feature with the highest correlation with diabetes
print(target_corr.idxmax())

glucose_conc


In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("posa_shaktoi.csv")

# Replace invalid 0 values with NaN
zero_cols = ["glucose_conc", "diastolic_bp", "thickness", "insulin", "bmi"]
df[zero_cols] = df[zero_cols].replace(0, np.nan)

# Impute missing values using median
imputer = SimpleImputer(strategy="median")
df[zero_cols] = imputer.fit_transform(df[zero_cols])

# Check missing values after imputation
print(df.isnull().sum())

# Separate features and target
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale feature values
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

num_preg        0
glucose_conc    0
diastolic_bp    0
thickness       0
insulin         0
bmi             0
diab_pred       0
age             0
diabetes        0
dtype: int64


In [35]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score, roc_auc_score, confusion_matrix


# Load dataset
df = pd.read_csv("posa_shaktoi.csv")

# Separate features and target
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train logistic regression model
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print (f"Accuracy: {accuracy:.4f}")
print (f"Precision: {precision:.4f}")
print (f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.7143
Precision: 0.6087
Recall: 0.5185
F1 Score: 0.5600
ROC AUC Score: 0.6693
Confusion Matrix:
[[82 18]
 [26 28]]


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
# use pipline for preprocessing pipline means 
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv("posa_shaktoi.csv")

# Replace invalid 0 values with NaN
zero_cols = ["glucose_conc", "diastolic_bp", "thickness", "insulin", "bmi"]
df[zero_cols] = df[zero_cols].replace(0, np.nan)

# Separate features and target
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Pipeline for XGBoost
xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ))
])

# Hyperparameter tuning
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 4, 5],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Best model after tuning
best_model = grid_search.best_estimator_

# Prediction
y_pred = best_model.predict(X_test)

# Evaluation
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(round(grid_search.best_score_, 4))

print("\nTest Accuracy:")
print(round(accuracy_score(y_test, y_pred), 4))

print("\nPrecision:")
print(round(precision_score(y_test, y_pred), 4))

print("\nRecall:")
print(round(recall_score(y_test, y_pred), 4))

print("\nF1 Score:")
print(round(f1_score(y_test, y_pred), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [54]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Load dataset
df = pd.read_csv("posa_shaktoi.csv")

# Replace invalid 0 values with NaN
zero_cols = ["glucose_conc", "diastolic_bp", "thickness", "insulin", "bmi"]
df[zero_cols] = df[zero_cols].replace(0, np.nan)

# Separate features and target
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Random Forest model
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

# Train model
rf_model.fit(X_train, y_train)

# Get feature importance
feature_importance = rf_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": feature_importance
})

importance_df = importance_df.sort_values(by="Importance", ascending=False)

print(importance_df)
print("\nTop 3 important features:")
print(importance_df.head(3))

        Feature  Importance
1  glucose_conc    0.262675
5           bmi    0.164553
7           age    0.127004
6     diab_pred    0.119491
4       insulin    0.097159
2  diastolic_bp    0.082443
0      num_preg    0.073424
3     thickness    0.073250

Top 3 important features:
        Feature  Importance
1  glucose_conc    0.262675
5           bmi    0.164553
7           age    0.127004


In [50]:
# advance model best performance SVM, random forest, xgboost, other

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv("posa_shaktoi.csv")

# Replace invalid 0 values with NaN
zero_cols = ["glucose_conc", "diastolic_bp", "thickness", "insulin", "bmi"]
df[zero_cols] = df[zero_cols].replace(0, np.nan)

# Separate features and target
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Models
models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]),

    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]),

    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=1,
            gamma="scale",
            random_state=42
        ))
    ])
}

# Train and evaluate models
best_model = None
best_accuracy = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(name)
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))
    print()

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = name

print("Best advanced model:", best_model)
print("Best test accuracy:", round(best_accuracy, 4))


Random Forest
Accuracy: 0.7403
Precision: 0.6522
Recall: 0.5556
F1 Score: 0.6

XGBoost
Accuracy: 0.7597
Precision: 0.6667
Recall: 0.6296
F1 Score: 0.6476

SVM
Accuracy: 0.7403
Precision: 0.6522
Recall: 0.5556
F1 Score: 0.6

Best advanced model: XGBoost
Best test accuracy: 0.7597
